# 15-Parameter Shortwave Calibration (Thesis Reproduction)

Reproduces the **multi-parameter shortwave calibration** from Chapter 5 of the thesis
("Multi-Parameter calibration", Table 5.4 / `tab:sw_params_15`), using the
`SpeedyCalibration.jl` package API. The original run was performed with the
ad-hoc `train_multi_param_online_v5` function in
`differentiability/main/point_wise_loss/5param_osr_temp_train.ipynb`
(the `training_params_15` block); this notebook is the equivalent workflow
using the packaged `calibrate!` API.

**Goal**: jointly tune 15 parameters spanning cloud reflection, atmospheric
absorption, and surface albedo against the Trenberth (2009) shortwave energy
budget:

| Flux | Target | Description |
|------|--------|--------------|
| OSR | 101.9 W/m² | Outgoing shortwave radiation (TOA, reflected) |
| SRU | 23.1 W/m²  | Surface shortwave up (reflected) |
| SRD | 184.3 W/m² | Surface shortwave down |

**Method**: online statistical gradient estimation — average single-timestep
Enzyme gradients over a continuously-running simulation (no differentiation
through the full trajectory), exactly as in the thesis.

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))   # SpeedyCalibration.jl project
Pkg.instantiate()

using SpeedyCalibration
using CairoMakie
using GeoMakie
using Optimisers
using Dates
using Printf

println("SpeedyCalibration.jl ready.")

## 1. Define the 15 trainable parameters

Groups and bounds are copied verbatim from Table 5.4 of the thesis
(`training_params_15` in `5param_osr_temp_train.ipynb`):

| Group | Name | Bounds | Initial |
|---|---|---|---|
| Cloud | `cloud_albedo` | [0.25, 0.95] | 0.60 |
| Cloud | `stratocumulus_cover_max` | [0.25, 0.95] | 0.60 |
| Cloud | `stratocumulus_albedo` | [0.10, 0.90] | 0.50 |
| Cloud cover | `precipitation_weight` | [0.0, 0.8] | 0.20 |
| Atm. absorption | `absorptivity_water_vapor` | [60, 140] | 75 |
| Atm. absorption | `absorptivity_dry_air` | [0.005, 0.060] | 0.031 |
| Atm. absorption | `absorptivity_aerosol` | [0.005, 0.060] | 0.031 |
| Atm. absorption | `ozone_absorption` | [0.002, 0.020] | 0.010 |
| Land albedo | `albedo_land` | [0.10, 0.70] | 0.40 |
| Land albedo | `albedo_high_vegetation` | [0.04, 0.26] | 0.15 |
| Land albedo | `albedo_low_vegetation` | [0.05, 0.35] | 0.20 |
| Land albedo | `albedo_snow` | [0.15, 0.75] | 0.40 |
| Land albedo | `snow_depth_scale` | [0.005, 0.20] | 0.050 |
| Ocean/ice albedo | `albedo_ocean` | [0.02, 0.10] | 0.06 |
| Ocean/ice albedo | `albedo_ice` | [0.30, 0.90] | 0.60 |

In [ ]:
param_specs = [
    # ── Cloud reflection (3) ──────────────────────────────────────────────────
    ParamSpec(:cloud_albedo,
        [:shortwave_radiation, :clouds, :cloud_albedo];
        bounds=(0.25f0, 0.95f0), initial=0.60f0),

    ParamSpec(:stratocumulus_cover_max,
        [:shortwave_radiation, :clouds, :stratocumulus_cover_max];
        bounds=(0.25f0, 0.95f0), initial=0.60f0),

    ParamSpec(:stratocumulus_albedo,
        [:shortwave_radiation, :clouds, :stratocumulus_albedo];
        bounds=(0.10f0, 0.90f0), initial=0.50f0),

    # ── Cloud cover (1) ───────────────────────────────────────────────────────
    ParamSpec(:precipitation_weight,
        [:shortwave_radiation, :clouds, :precipitation_weight];
        bounds=(0.0f0, 0.8f0), initial=0.20f0),

    # ── Atmospheric absorption (4) ───────────────────────────────────────────
    # absorptivity_water_vapor has a ~200× weaker physical gradient than the
    # cloud parameters (it is multiplied by specific humidity q ≈ 0.005 in the
    # radiation scheme). Lower bound raised to 60 (from 10): the model becomes
    # numerically unstable below ~57.
    ParamSpec(:absorptivity_water_vapor,
        [:shortwave_radiation, :transmissivity, :absorptivity_water_vapor];
        bounds=(60f0, 140f0), initial=75f0),

    ParamSpec(:absorptivity_dry_air,
        [:shortwave_radiation, :transmissivity, :absorptivity_dry_air];
        bounds=(0.005f0, 0.060f0), initial=0.03135f0),

    ParamSpec(:absorptivity_aerosol,
        [:shortwave_radiation, :transmissivity, :absorptivity_aerosol];
        bounds=(0.005f0, 0.060f0), initial=0.03135f0),

    ParamSpec(:ozone_absorption,
        [:shortwave_radiation, :radiative_transfer, :ozone_absorption];
        bounds=(0.002f0, 0.020f0), initial=0.01f0),

    # ── Land surface albedo (5) ──────────────────────────────────────────────
    ParamSpec(:albedo_land,
        [:albedo, :land, :albedo_land];
        bounds=(0.10f0, 0.70f0), initial=0.40f0),

    ParamSpec(:albedo_high_vegetation,
        [:albedo, :land, :albedo_high_vegetation];
        bounds=(0.04f0, 0.26f0), initial=0.15f0),

    ParamSpec(:albedo_low_vegetation,
        [:albedo, :land, :albedo_low_vegetation];
        bounds=(0.05f0, 0.35f0), initial=0.20f0),

    ParamSpec(:albedo_snow,
        [:albedo, :land, :albedo_snow];
        bounds=(0.15f0, 0.75f0), initial=0.40f0),

    ParamSpec(:snow_depth_scale,
        [:albedo, :land, :snow_depth_scale];
        bounds=(0.005f0, 0.20f0), initial=0.05f0),

    # ── Ocean/ice surface albedo (2) ─────────────────────────────────────────
    ParamSpec(:albedo_ocean,
        [:albedo, :ocean, :albedo_ocean];
        bounds=(0.02f0, 0.10f0), initial=0.06f0),

    ParamSpec(:albedo_ice,
        [:albedo, :ocean, :albedo_ice];
        bounds=(0.30f0, 0.90f0), initial=0.60f0),
]

println("$(length(param_specs)) parameters defined:")
for spec in param_specs
    @printf("  %-28s  init=%-10.5f  bounds=[%.4f, %.4f]\n",
            spec.name, spec.initial, spec.bounds...)
end

## 2. Configure the loss function

Three-flux MSE loss (OSR + SRU + SRD) with **equal weights** ($w_{osr}=w_{sru}=w_{srd}=1$),
matching the thesis exactly (Section "Multi-Parameter calibration"):

$$\mathcal{L}^{(b)} = w_{osr}(\bar{\mu}_{OSR}^{(b)} - 101.9)^2 + w_{sru}(\bar{\mu}_{SRU}^{(b)} - 23.1)^2 + w_{srd}(\bar{\mu}_{SRD}^{(b)} - 184.3)^2$$

In [ ]:
# Targets from Wild et al. (2015) / Trenberth (2009) — same as the thesis run
sw_loss = LossConfig([:osr, :sru, :srd];
    targets = Dict(:osr => 101.9f0, :sru => 23.1f0, :srd => 184.3f0),
    weights = Dict(:osr => 1f0,     :sru => 1f0,     :srd => 1f0))

println("Loss: OSR=$(sw_loss.targets[:osr]) SRU=$(sw_loss.targets[:sru]) SRD=$(sw_loss.targets[:srd]) W/m²  (equal weights)")

## 3. Training configuration

Hyperparameters follow the thesis description as closely as the packaged API allows:

- `daily_cycle=true` — diurnal cycle **enabled** (unlike the 5-parameter baseline notebook), seasonal cycle disabled
- `start_date = 2000-03-21` — vernal equinox, so radiative forcing starts roughly symmetric between hemispheres
- `spinup_days=20` — 20-day spin-up before touching gradients
- `batch_days=30.0` — parameters updated every 30 simulated days
- `samples_per_batch=15` — 15 single-step gradient samples averaged per batch
- `max_batches=300` — thesis run converged at batch 81; this is a generous ceiling
- Adam learning rate `η=0.1`, gradients clipped before each step
- LR halved after 10 batches without improvement in the 20-batch smoothed loss, up to 5 times, floor `1e-5`
- Training stops once the 20-batch smoothed loss falls below `1.0`

In [ ]:
config = TrainingConfig(
    spinup_days       = 20,
    batch_days        = 30.0,
    samples_per_batch = 15,
    max_batches       = 300,
    loss_window_size  = 20,
    loss_threshold    = 1f0,      # converge when 20-batch smoothed loss < 1 W²/m⁴
    patience          = 10,
    grad_clip         = 5f0,
    enable_lr_decay   = true,
    lr_decay_factor   = 0.5f0,
    lr_plateau_patience = 10,
    min_lr            = 1f-5,
    max_lr_decays     = 5,
    trunc             = 31,
    nlayers           = 8,
    start_date        = DateTime(2000, 3, 21),
    daily_cycle       = true,
    verbose           = true,
)
println(config)

## 4. Run calibration

`calibrate!` handles everything:
1. Builds the model and sets initial parameter values
2. Runs the spinup
3. Warms up Enzyme's AD rules on the spun-up model
4. Runs the batch loop: sample gradients → average → Adam step → reconstruct model
5. Saves all artifacts to `save_dir/`

**Runtime**: with the diurnal cycle enabled and 15 parameters this is
substantially slower than the 5-parameter baseline — expect several hours at T31,
dominated by Enzyme precompilation on the first call.

In [ ]:
save_dir = joinpath(@__DIR__, "output", "thesis_15param_shortwave")

result = calibrate!(
    param_specs,
    Optimisers.Adam(1f-1),
    sw_loss,
    config;
    save_dir = save_dir,
)

## 5. Training summary

In [ ]:
println(result)
println("\nFinal parameter values:")
for spec in param_specs
    v = result.final_params[spec.name]
    Δ = v - spec.initial
    @printf("  %-28s  init=%.4f  final=%.4f  Δ=%+.4f\n",
            spec.name, spec.initial, v, Δ)
end

println("\nFinal flux means (last batch):")
for k in [:osr, :sru, :srd]
    v = isempty(result.history[k]) ? NaN32 : result.history[k][end]
    t = sw_loss.targets[k]
    @printf("  %s: %.2f W/m²  (target %.1f,  Δ=%+.2f)\n", k, v, t, v - t)
end

## 6. Plot training history

Reproduces Figure 5.x ("Training convergence of the 15-parameter shortwave
calibration") from the thesis: smoothed loss, OSR/SRU/SRD trajectories,
per-parameter trajectories, and gradient magnitudes.

In [ ]:
using GeoMakie

In [ ]:
figs = plot_training(result; save_dir = save_dir)
figs.fig_loss

In [ ]:
figs.fig_flux

In [ ]:
figs.fig_params

In [ ]:
figs.fig_grads

## 6b. Deck-styled export (slide 8)

Re-renders the loss/OSR/SRU/SRD convergence panels in the slide deck's own palette and
layout, replacing the digitised version currently on slide 8, as a single 1290×350 pt PNG.

`animate`/`plot_training` above can't produce this directly — the deck needs an exact
background colour, coastline-free line styling, specific fonts/sizes, and precise per-panel
axis ranges that aren't exposed as options, so this is a small standalone figure built
straight from `result.history` with `CairoMakie.Theme`. `colorant"..."` from the original
spec needs the `Colors` package, which isn't a dependency of this project, so plain hex
strings are used instead — Makie parses those identically.

**Reconciling against the real log** (the caveat from the spec): the digitised slide-8
panels trail off around **OSR ≈ 99.4 W/m²**, and the deck text quotes a final batch mean of
**101.6 W/m²**. This actual run's last batch (98) gives:

| | OSR | SRU | SRD |
|---|---|---|---|
| Digitised (slide 8) | ~99.4 | — | — |
| Deck text | 101.6 | — | — |
| **This run, batch 98** | **102.25** | **23.25** | **184.63** |
| Target | 101.9 | 23.1 | 184.3 |

None of the three numbers agree exactly — training is stochastic (15 online gradient
samples/batch) so re-running never reproduces a prior run bit-for-bit. Use the real
regenerated figure below and update the deck text to whatever this run's own final numbers
are, rather than trying to match the old digitised curve.

In [ ]:
const BG   = "#F3F2F2"
const INK  = "#201E1D"
const GREY = "#4A4644"
const RED  = "#AE1800"
const HAIR = "#C9C6C4"

deck_theme = Theme(
    backgroundcolor = :transparent,
    fonts = (regular = "Arial", bold = "Arial Bold", mono = "Courier New"),
    fontsize = 14,
    figure_padding = 14,
    Axis = (
        backgroundcolor    = :transparent,
        titlefont = :regular, titlesize = 16, titlecolor = INK, titlealign = :left,
        xlabelsize = 14, ylabelsize = 14, xlabelcolor = GREY, ylabelcolor = GREY,
        xticklabelsize = 14, yticklabelsize = 14,
        xticklabelcolor = GREY, yticklabelcolor = GREY,
        xgridvisible = false, ygridvisible = true,
        ygridcolor = HAIR, ygridwidth = 0.8,
        topspinevisible = false, rightspinevisible = false,
        leftspinecolor = INK, bottomspinecolor = INK,
        leftspinewidth = 1, bottomspinewidth = 1,
        xtickcolor = INK, ytickcolor = INK, xtickwidth = 1, ytickwidth = 1,
        xticksize = 4, yticksize = 4, xminorticksvisible = false,
        titlegap = 6,
        xlabelpadding = 4,
        ylabelpadding = 4,
    ),
    Lines  = (linewidth = 2.5,),
    Legend = (framevisible = false, labelsize = 14, labelcolor = INK,
              patchsize = (18, 2), padding = (0, 0, 0, 0)),
)

function plot_slide8_convergence(result; output_path)
    h = result.history
    batch = h[:batch]
    bmax = 20 * cld(maximum(batch), 20)   # round up to the next multiple of 20

    # y-limits sized with a margin above/below the real data's actual extrema,
    # not the fixed 78-106/17-26/180-202 from the original digitised-figure spec --
    # those clipped this run's real peaks (e.g. OSR spikes to ~109 around batch 15).
    pad(v, lo, hi) = (min(lo, floor(minimum(v) - 1)), max(hi, ceil(maximum(v) + 1)))
    loss_lims = (1, 1000)
    osr_lims  = pad(h[:osr], 78, 106)
    sru_lims  = pad(h[:sru], 17, 26)
    srd_lims  = pad(h[:srd], 180, 202)

    xt = 0:20:bmax

    with_theme(deck_theme) do
        fig = Figure(size = (1340, 360))

        ax1 = Axis(fig[1, 1]; title = "Smoothed loss", xlabel = "Batch",
                   yscale = log10, xticks = xt)
        xlims!(ax1, 0, bmax)
        ylims!(ax1, loss_lims...)
        lines!(ax1, batch, h[:smoothed_loss]; color = RED, linewidth = 2.5)

        ax2 = Axis(fig[1, 2]; title = "OSR", ylabel = "W m⁻²", xticks = xt)
        xlims!(ax2, 0, bmax)
        ylims!(ax2, osr_lims...)
        lines!(ax2, batch, h[:osr]; color = RED, linewidth = 2.5)
        hlines!(ax2, [101.9]; color = INK, linestyle = :dash, linewidth = 1.5)

        ax3 = Axis(fig[1, 3]; title = "SRU", xticks = xt)
        xlims!(ax3, 0, bmax)
        ylims!(ax3, sru_lims...)
        lines!(ax3, batch, h[:sru]; color = RED, linewidth = 2.5)
        hlines!(ax3, [23.1]; color = INK, linestyle = :dash, linewidth = 1.5)

        ax4 = Axis(fig[1, 4]; title = "SRD", xticks = xt)
        xlims!(ax4, 0, bmax)
        ylims!(ax4, srd_lims...)
        lines!(ax4, batch, h[:srd]; color = RED, linewidth = 2.5)
        hlines!(ax4, [184.3]; color = INK, linestyle = :dash, linewidth = 1.5)

        save(output_path, fig; px_per_unit = 4)
        return fig
    end
end

slide8_path = joinpath(save_dir, "slide8_convergence.png")
plot_slide8_convergence(result; output_path = slide8_path)

## 7. Climate validation

Compare a 7-year run with **default** vs. **trained** parameters, with
equilibrium statistics over the last 5 years (matching the thesis's
7-year / final-6-year evaluation as closely as the packaged API allows).

In [ ]:
validation = run_climate_validation(result; n_years=7, stat_years=5, dt=Minute(20))


In [ ]:
# Equilibrium budget comparison (reproduces Table 5.x / tab:15_param_tuning_table)
println("\nEquilibrium budget (mean over last 5 years):")
println("-" ^ 52)
@printf("%-12s  %10s  %10s  %10s\n", "Flux", "Default", "Trained", "Target")
println("-" ^ 52)
for (k, key, label) in [
        (:osr, :osr, "OSR [W/m²]"),
        (:sru, :sru, "SRU [W/m²]"),
        (:srd, :srd, "SRD [W/m²]"),
        (:olr, :olr, "OLR [W/m²]"),
    ]
    d = getfield(validation.default, key)
    t = getfield(validation.trained, key)
    tgt = get(sw_loss.targets, k, NaN32)
    @printf("%-12s  %10.2f  %10.2f  %10s\n", label, d, t,
            isnan(tgt) ? "—" : @sprintf("%.1f", tgt))
end
println("-" ^ 52)

In [ ]:
cfigs = plot_climate(validation; save_dir = save_dir, loss_config = sw_loss)
save(joinpath(save_dir, "fig_clm_rad.png"), cfigs.fig_rad; px_per_unit = 4)
cfigs.fig_rad

In [ ]:
cfigs.fig_summary

## 7b. Climate validation with the seasonal cycle turned on (exploratory)

The run above uses `seasonal_cycle=false` (perpetual equinox), the setting shared by every
notebook in this directory — it's held constant deliberately so that different calibration
experiments stay comparable (see `examples/README.md`, "Why not seasonal cycle next"). This
15-parameter SW-only calibration was **trained** under that same setting and is not being
retrained here — only the post-hoc climate validation is repeated with a real annual cycle, to
see how much the equilibrium fluxes shift once the sun actually moves. Since `_build_model`
previously hardcoded `seasonal_cycle=false`, `SpeedyCalibration.jl`'s `run_climate_validation`
now takes a `seasonal_cycle` keyword (default `false`, unchanged for every other caller).

In [ ]:
validation_seasonal = run_climate_validation(result; n_years=7, stat_years=5, dt=Minute(20), seasonal_cycle=true)

In [ ]:
# Compare: perpetual-equinox validation (above) vs. real seasonal cycle
println("\nEquilibrium budget, seasonal_cycle=true (mean over last 5 years):")
println("-" ^ 64)
@printf("%-12s  %10s  %10s  %10s  %10s\n", "Flux", "Default", "Trained", "No-season", "Target")
println("-" ^ 64)
for (k, key, label) in [
        (:osr, :osr, "OSR [W/m²]"),
        (:sru, :sru, "SRU [W/m²]"),
        (:srd, :srd, "SRD [W/m²]"),
        (:olr, :olr, "OLR [W/m²]"),
    ]
    d  = getfield(validation_seasonal.default, key)
    t  = getfield(validation_seasonal.trained, key)
    t0 = getfield(validation.trained, key)
    tgt = get(sw_loss.targets, k, NaN32)
    @printf("%-12s  %10.2f  %10.2f  %10.2f  %10s\n", label, d, t, t0,
            isnan(tgt) ? "—" : @sprintf("%.1f", tgt))
end
println("-" ^ 64)
println("(\"No-season\" = trained-parameter result from cell 7, seasonal_cycle=false)")

In [ ]:
cfigs_seasonal = plot_climate(validation_seasonal; save_dir = save_dir, loss_config = sw_loss)
save(joinpath(save_dir, "fig_clm_rad_seasonal.png"), cfigs_seasonal.fig_rad; px_per_unit = 4)
cfigs_seasonal.fig_rad

## Summary

This notebook reproduces the thesis's multi-parameter shortwave calibration workflow:

1. **15 parameters** across cloud reflection (3), cloud cover (1), atmospheric
   absorption (4), land albedo (5), and ocean/ice albedo (2)
2. **OSR + SRU + SRD loss** with equal weights and Trenberth (2009) targets
3. **20-day spinup** + batches of 15 gradient samples every 30 simulated days,
   with the diurnal cycle enabled
4. **Full climate validation** to confirm improvements at 7-year timescales

Results are saved to `examples/output/thesis_15param_shortwave/`:
- `result.jld2` — full `TrainingResult` (resumable)
- `history.csv` — per-batch training history
- `final_weights.toml` — human-readable final parameter values
- `fig_*.pdf`, `climate_*.pdf` — figures matching the thesis